# DSE 200 — Day 4 Lab
## Advanced Models and Production

**Agent on — explain every cell**

Section numbers match the slides.

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while Rick talks. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost — every section rebuilds its own data. This file has 1.5 million rows, so some cells take a minute.

In [ ]:
# ▶ SETUP — run first. Finds the data and defines check_read() and the weather helpers.
import os, sys, subprocess, textwrap, importlib.util, time
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

if importlib.util.find_spec("xgboost") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
from xgboost import XGBClassifier

def fetch(name):
    """Find a course data file, or download it. Opening from GitHub brings the notebook, not the data."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
    print("downloaded", name, "from the course repo")
    return name

WEATHER = fetch("minute_weather.csv.gz")
CUSTOMERS = fetch("customers.csv")

SENSORS = ["air_pressure", "air_temp", "avg_wind_direction", "avg_wind_speed",
           "max_wind_direction", "max_wind_speed", "min_wind_direction",
           "min_wind_speed", "rain_accumulation", "rain_duration", "relative_humidity"]
BACKWARD = ["hour", "month", "rh_change_1h", "rh_change_3h", "temp_change_1h", "pressure_change_3h"]

_loaded = None
def weather():
    """The decisions from 8.3 and 8.4, in one place: no copied rows, and the dry_now / dry_3h columns."""
    global _loaded
    if _loaded is None:
        d = pd.read_csv(WEATHER, parse_dates=["hpwren_timestamp"])
        d = d.drop_duplicates(subset=d.columns.drop("rowID"))
        d["dry_now"] = (d["relative_humidity"] < 25).astype(int)
        d["dry_3h"] = (d["relative_humidity"].shift(-180) < 25).astype(int)
        _loaded = d.iloc[:-180]
    return _loaded.copy()

def add_features(d):
    """The features from 8.7. Every one looks backward."""
    d = d.copy()
    d["hour"] = d["hpwren_timestamp"].dt.hour
    d["month"] = d["hpwren_timestamp"].dt.month
    d["rh_change_1h"] = d["relative_humidity"] - d["relative_humidity"].shift(60)
    d["rh_change_3h"] = d["relative_humidity"] - d["relative_humidity"].shift(180)
    d["temp_change_1h"] = d["air_temp"] - d["air_temp"].shift(60)
    d["pressure_change_3h"] = d["air_pressure"] - d["air_pressure"].shift(180)
    return d

def three_years(d):
    """Train, decide when to stop, sit the exam. Whole years, so every season is in every split."""
    t = d["hpwren_timestamp"]
    return d[t < "2012-09-10"], d[(t >= "2012-09-10") & (t < "2013-09-10")], d[t >= "2013-09-10"]

def errors(y, pred):
    """Accuracy, and the two kinds of wrong."""
    y, pred = np.asarray(y), np.asarray(pred)
    return (y == pred).mean(), int(((y == 1) & (pred == 0)).sum()), int(((y == 0) & (pred == 1)).sum())

_READ = {
"A.1": ("Occasional customers, even though their accuracy is 0.849.",
        "The next cell shows it: the tree misses 121 of the 139 occasional customers who came\n"
        "back. Only 14.5% of that group return, so saying 'no' nearly every time scores well.\n"
        "It is week 3's 'always no' trick, inside one group, and accuracy cannot show it."),
"8.2": ("No. It ties the forest (0.813 against 0.814) and loses to the depth-4 tree (0.825).",
        "Last week's slide promised boosting would beat both. On this file it does not.\n"
        "But 0.823 and 0.825 are only 4 customers apart on one test set, so one split\n"
        "cannot settle it either. Boosting usually wins on tables. 'Usually' is why you compare."),
"8.5": ("No -- and nothing in that code can tell you why.",
        "The held-out rows are minutes that sit between training minutes. Humidity typically\n"
        "moves 0.2 points a minute, and the next minute's dry_3h label is the same 99% of the\n"
        "time, so the model can copy a neighbor's answer. In real use, no minutes from the\n"
        "future are in the training data.\n"
        "Next cell: the same model, tested on a year it has not seen."),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print(f"data: {WEATHER}, {CUSTOMERS}")
print("check_read(), weather(), add_features(), three_years() and errors() ready.")

# Who It Fails, And What Would Change It

*Slides D04.02 · sections A.1–A.5 · audit a model by group, then run an A/B test · exercises A.2 and A.3*

Worth doing even if we don't reach it in class: HW4's audit deliverable uses this method.

## A.1 The Audit

Last week's best model, a depth-4 tree, scores 0.825. Who is behind that number?

In [ ]:
# ▶ SETUP A.1 — from cold: last week's customers, split and tree, plus the groups to audit
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

FEATURES = ["n_orders", "total_spend", "avg_order_value",
            "days_since_last_order", "tenure_days", "n_categories"]
c = pd.read_csv(CUSTOMERS).dropna(subset=FEATURES)
c = c[(c.total_spend >= 0) & (c.n_orders > 0) & (c.tenure_days <= 3650)].copy()
c["avg_order_value"] = (c.total_spend / c.n_orders).round(2)

# last week's four segments, named by what they look like
Xs = StandardScaler().fit_transform(c[["total_spend", "n_orders", "days_since_last_order"]])
c["cluster"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Xs).labels_
m = c.groupby("cluster")[["total_spend", "days_since_last_order"]].median()
lapsed, vips = m.days_since_last_order.idxmax(), m.total_spend.idxmax()
others = m.drop([lapsed, vips]).total_spend.sort_values().index
c["segment"] = c.cluster.map({lapsed: "lapsed", vips: "VIPs", others[0]: "occasional", others[1]: "regulars"})
c["tenure"] = pd.cut(c.tenure_days, [0, 365, 730, 1100, 4000],
                     labels=["under 1 year", "1-2 years", "2-3 years", "3+ years"])

train, test = train_test_split(c, test_size=0.2, random_state=42, stratify=c["returned"])
tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(train[FEATURES], train["returned"])
test = test.assign(pred=tree.predict(test[FEATURES]))
print("accuracy", round((test["pred"] == test["returned"]).mean(), 3),
      "  missed", int(((test["returned"] == 1) & (test["pred"] == 0)).sum()), "of", int(test["returned"].sum()))

In [ ]:
# ▶ DEMO -- the audit is one table
def audit(test, col):
    def row(g):
        came_back = g[g["returned"] == 1]
        return pd.Series({"customers": len(g), "came_back": len(came_back),
                          "accuracy": (g["pred"] == g["returned"]).mean(),
                          "missed": (came_back["pred"] == 0).mean()})
    return test.groupby(col, observed=True).apply(row)

print(audit(test, "segment")[["customers", "accuracy"]].round(3))

In [ ]:
# 🔍 READ A.1 — which group does the model fail most? You have only seen accuracy.
# Commit first.

prediction = "???"
check_read("A.1", prediction)

In [ ]:
# ▶ DEMO -- now with the column accuracy was hiding
print(audit(test, "segment").round(3))

**Occasional customers score 85%, and the model misses 87% of those who come back.** `missed` is the share of that group's returners the model said "no" to. Lapsed customers show a blank because none of them came back.

## A.2 Your Turn — 15 minutes

Audit three more ways. The model never saw `channel` or `region`. Then decide which gaps you believe: slice enough ways and something will always look bad, so test each one.

In [ ]:
# ▶ SETUP A.2 -- the shuffle test from Week 1, for a gap in `missed`
def miss_gap_share(test, col, n=300):
    # how often do shuffled group labels give a gap in `missed` at least as big as the real one?
    d = test.dropna(subset=[col])
    back = (d["returned"] == 1).to_numpy()
    def gap(labels):
        r = d[back].assign(label=labels[back]).groupby("label", observed=True)["pred"].apply(lambda p: (p == 0).mean())
        return r.max() - r.min()
    real = gap(d[col].to_numpy())
    rng = np.random.default_rng(0)
    return np.mean([gap(rng.permutation(d[col].to_numpy())) >= real for _ in range(n)])

In [ ]:
# ✍️ YOU TRY A.2
# 1. worst_channel, worst_tenure, worst_region -- the group with the highest `missed` in each
#       audit(test, "channel")["missed"].idxmax()
# 2. store_missed -- the `missed` value for store customers
# 3. channel_share, tenure_share, region_share -- miss_gap_share(test, col) for each column

worst_channel = worst_tenure = worst_region = ""      # TODO
store_missed = 0.0                                    # TODO
channel_share = tenure_share = region_share = 1.0     # TODO

print(worst_channel, worst_tenure, worst_region, round(store_missed, 3))
print(channel_share, tenure_share, region_share)

In [ ]:
# ✅ CHECKPOINT A.2
assert worst_channel == "store", worst_channel
assert worst_tenure == "under 1 year", worst_tenure
assert worst_region == "East", worst_region
assert abs(store_missed - 0.654) < 0.002, store_missed
assert channel_share < 0.02, channel_share
assert tenure_share < 0.05, tenure_share
assert region_share > 0.5, region_share
print("Correct.\n")
print(f"Channel and tenure hold up (shuffle share {channel_share:.1%} and {tenure_share:.1%}).")
print(f"Region does not ({region_share:.0%}). Pacific has only 5 returners in the test set.")

**You cannot fix every number at once.** Compare return rate, `missed` and false alarms by segment: low-return groups get missed and high-return groups get false alarms. When base rates differ, a model that means the same thing by a score cannot also have equal error rates, so someone must choose which error matters for whom.

## A.3 Did Your Agent Check? — 5 minutes

Give your agent this and nothing more:

> *"Build a model that predicts `returned` from this customer file, and tell me how good it is."*

Did it report anything by group? Did it say who the model fails, without being asked? If not, what did its summary lead you to believe?

**Write your two sentences here** (double-click to edit), to whoever owns this model:

1. Who it fails: the group, the number, and why you believe it isn't luck.
2. What you'd do: change the threshold for them, get better data, or tell the people using it. Say which, and what it costs.

*(your answer)*

## A.4 An A/B Test

The audit says who. It can't say whether a coupon would bring them back: the model describes customers who never got one. That takes an experiment, and a coin decides who gets the coupon.

**These numbers are simulated:** the coupon truly lifts returning from 10% to 12%.

In [ ]:
# ▶ DEMO -- run one
rng = np.random.default_rng(7)
control = rng.binomial(1, 0.10, 1000)     # 1 = came back. 1,000 customers, no coupon
coupon  = rng.binomial(1, 0.12, 1000)     # 1,000 customers with the coupon
print(control.mean(), coupon.mean())

In [ ]:
# ▶ DEMO -- is it luck? Pretend the coupon did nothing, and relabel at random
gap = coupon.mean() - control.mean()
both = np.concatenate([control, coupon])
fake = [np.diff([s[:1000].mean(), s[1000:].mean()])[0]
        for s in (rng.permutation(both) for _ in range(2000))]
print("gap", round(gap, 3), "  shuffles at least this big", np.mean(np.abs(fake) >= abs(gap)))

In [ ]:
# ▶ DEMO -- how big? Resample each group, with repeats, for a 95% confidence interval
boot = [rng.choice(coupon, 1000).mean() - rng.choice(control, 1000).mean()
        for _ in range(2000)]
print("95% interval for the lift:", np.percentile(boot, [2.5, 97.5]).round(3))

## A.5 How A/B Tests Mislead

Run the whole experiment thousands of times, in a world where you know the truth.

In [ ]:
# ▶ DEMO -- too few customers: how often does the test detect a real lift from 10% to 12%?
sim = np.random.default_rng(1)
def significant(n, pa, pb, sims=20000):
    a, b = sim.binomial(n, pa, sims) / n, sim.binomial(n, pb, sims) / n
    p = (a + b) / 2
    return np.abs(b - a) / np.maximum(np.sqrt(2 * p * (1 - p) / n), 1e-9) > 1.96

for n in [500, 1000, 2000, 4000, 8000]:
    print(f"{n:>5} per group   detects the real lift {significant(n, 0.10, 0.12).mean():.0%}   false win with no lift {significant(n, 0.10, 0.10).mean():.0%}")

In [ ]:
# ▶ DEMO -- peeking: a coupon that does nothing, checked every 100 customers up to 2,000 per group
sim = np.random.default_rng(3)
a = sim.binomial(1, 0.10, (5000, 2000)).cumsum(axis=1)
b = sim.binomial(1, 0.10, (5000, 2000)).cumsum(axis=1)
looks = np.arange(100, 2001, 100)
pa, pb = a[:, looks - 1] / looks, b[:, looks - 1] / looks
p = (pa + pb) / 2
hit = np.abs(pa - pb) / np.maximum(np.sqrt(2 * p * (1 - p) / looks), 1e-9) > 1.96
print(f"read once at the end: {hit[:, -1].mean():.0%} false wins")
print(f"read at every look:   {hit.any(axis=1).mean():.0%} false wins")

**Fix the sample size and the metric before you start, and read it once.** And name the decision in advance: a coupon can lift returning and still lose money, because it also went to people who would have come back anyway.

**The model finds who. The experiment finds what works.**

# Gradient Boosting

*Slides D04.03 · sections 8.1–8.9 · one loop, three knobs, and a score you should not believe · exercise 8.9*

## 8.1 One Loop

| | Forest | Boosting |
| --- | --- | --- |
| trees are built | independently | one after another |
| each tree sees | a random sample | the errors so far |
| each tree is | deep | small |
| the answer | a vote | a sum |

Boosting fits in six lines. `X` is the day of the year; `y` is that day's mean temperature, 2013.

In [ ]:
# ▶ SETUP 8.1 — from cold
from sklearn.tree import DecisionTreeRegressor
d = weather()
d = d[(d["hpwren_timestamp"] >= "2013-01-01") & (d["hpwren_timestamp"] < "2014-01-01")]
daily = d.groupby(d["hpwren_timestamp"].dt.dayofyear)["air_temp"].mean()
X = daily.index.to_numpy().reshape(-1, 1)
y = daily.to_numpy()
print(len(y), "days   mean", round(y.mean(), 1))

In [ ]:
# ▶ DEMO -- boosting by hand
pred = np.full(len(y), y.mean())          # start: everyone gets the average
for i in range(1, 101):
    errors_now = y - pred                 # what is still wrong?
    tree = DecisionTreeRegressor(max_depth=1).fit(X, errors_now)   # one tiny tree
    pred = pred + 0.3 * tree.predict(X)   # add a little of its answer
    if i in (1, 5, 20, 100):
        print(f"after {i:>3} trees   squared error {((y - pred) ** 2).mean():.1f}")

plt.scatter(X, y, s=6, alpha=0.5)
plt.step(X.ravel(), pred, where="mid", color="tab:orange")
plt.xlabel("day of 2013"); plt.ylabel("mean air temp (F)")
plt.title("100 one-question trees, added together")
plt.show()

Change `range(1, 101)` to `range(1, 6)` and run it again. Then change `0.3` to `1.0`. **That number is `learning_rate`; `max_depth=1` is the tiny tree.** XGBoost is this loop, built to be fast, with one more idea: for a yes/no target, the trees add up a score and last week's S-curve turns it into a chance.

## 8.2 XGBoost

Last week's customer file and last week's split, so we can compare. The same four lines as every other model.

In [ ]:
# ▶ SETUP 8.2 — from cold: the cleaned customer table and the split from 7.1
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

FEATURES = ["n_orders", "total_spend", "avg_order_value",
            "days_since_last_order", "tenure_days", "n_categories"]
c = pd.read_csv(CUSTOMERS).dropna(subset=FEATURES)
c = c[(c.total_spend >= 0) & (c.n_orders > 0) & (c.tenure_days <= 3650)].copy()
c["avg_order_value"] = (c.total_spend / c.n_orders).round(2)
X_train, X_test, y_train, y_test = train_test_split(
    c[FEATURES], c["returned"], test_size=0.2, random_state=42, stratify=c["returned"])
print(len(X_train), "train  ", len(X_test), "test  ", int(y_test.sum()), "returners in test")

In [ ]:
# ▶ DEMO
xgb = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
xgb.fit(X_train, y_train)
xgb.score(X_test, y_test)

In [ ]:
# 🔍 READ 8.2 — did boosting win?
#
#     always "no"              0.777
#     logistic regression      0.823
#     decision tree, depth 4   0.825
#     random forest            0.814
#     xgboost                  0.813
#
# Last week's slide said boosting would beat the tree and the forest. Did it? Commit.

prediction = "???"
check_read("8.2", prediction)

In [ ]:
# ▶ DEMO -- five tests instead of one (about half a minute)
folds = StratifiedKFold(5, shuffle=True, random_state=42)
models = {
    "logistic regression":    LogisticRegression(max_iter=1000),
    "decision tree, depth 4": DecisionTreeClassifier(max_depth=4, random_state=42),
    "xgboost, gentler":       XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42),
    "random forest":          RandomForestClassifier(n_estimators=300, random_state=42),
    "xgboost":                xgb,
}
for name, model in models.items():
    s = cross_val_score(model, X_train, y_train, cv=folds)
    print(f"{name:24} {s.mean():.3f}  +- {s.std():.3f}")

**The simplest model comes first.** On this file the signal is simple: the depth-4 tree leans almost entirely on two features, so there is little left for boosting to find. Boosting usually wins on tables. "Usually" is why you compare.

## 8.3 A Bigger File

One minute per row, three years, a San Diego weather station. The same first five minutes as every other file.

In [ ]:
# ▶ DEMO
df = pd.read_csv(WEATHER, parse_dates=["hpwren_timestamp"])
print(df.shape)
print(df.isna().sum()[lambda s: s > 0], "\n")
df.describe().T[["min", "50%", "max"]].round(1)

In [ ]:
# ▶ DEMO -- how many times does each moment appear?
df["hpwren_timestamp"].value_counts().head(2)

**One second of April 2012 appears 32,523 times.** It looks like a stuck logger. Now look at what last week's check would have said.

In [ ]:
# ▶ DEMO -- every copy has its own ID
print("duplicated rowIDs:", df["rowID"].duplicated().sum())
df = df.drop_duplicates(subset=df.columns.drop("rowID"))
print("rows left:", len(df))

**32,522 exact copies, each with a fresh `rowID`.** Split the file as it stands and the same row lands in train *and* test, and `set(X_train.index) & set(X_test.index)` comes back empty. That check only compares IDs.

## 8.4 A Real Question

**Will humidity be under 25% three hours from now?**

Imagine a crew planning a controlled burn this afternoon. A false alarm cancels a day's work. A miss could burn a hillside.

In [ ]:
# ▶ DEMO -- the label is a column from the future
df["dry_now"] = (df["relative_humidity"] < 25).astype(int)
df["dry_3h"]  = (df["relative_humidity"].shift(-180) < 25).astype(int)
df = df.iloc[:-180].copy()            # the last 3 hours have no answer yet
print("dry in 3 hours:", round(df["dry_3h"].mean(), 3))

later = df["hpwren_timestamp"].shift(-180) - df["hpwren_timestamp"]
print("rows where 180 rows ahead is 175-185 minutes:", f"{later.dt.total_seconds().div(60).between(175, 185).mean():.2%}")

`shift(-180)` reads the value 180 rows **ahead**. It is only allowed in the label. And it assumes one row per minute, so we checked: the file has a few gaps, and 99.66% of rows land within five minutes of three hours.

## 8.5 The Wrong Exam

Exactly what you did last week: a seed, stratified, no row in both halves.

In [ ]:
# ▶ SETUP 8.5 — from cold
from sklearn.model_selection import train_test_split
df = weather()
print(df.shape)

In [ ]:
# ▶ DEMO -- a random split, and a big model (about a minute)
X_tr, X_te, y_tr, y_te = train_test_split(df[SENSORS], df["dry_3h"],
    test_size=0.2, random_state=42, stratify=df["dry_3h"])
deep = XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=12, random_state=42)
random_score = deep.fit(X_tr, y_tr).score(X_te, y_te)
print("always no                  ", round(1 - y_te.mean(), 3))
print("model, on the held-out rows", round(random_score, 3))

In [ ]:
# 🔍 READ 8.5 — would you ship this model?
#
# 0.958 against a baseline of 0.740. A seed, stratified, no row in both halves.
# Yes or no? Commit.

prediction = "???"
check_read("8.5", prediction)

In [ ]:
# ▶ DEMO -- the exam it will actually sit: train on the past, test on the future (about a minute)
past = df[df["hpwren_timestamp"] < "2013-09-10"]
year3 = df[df["hpwren_timestamp"] >= "2013-09-10"]
deep.fit(past[SENSORS], past["dry_3h"])
print("on its own training years", round(deep.score(past[SENSORS], past["dry_3h"]), 3))
print("on the final year         ", round(deep.score(year3[SENSORS], year3["dry_3h"]), 3))

In [ ]:
# ▶ DEMO -- and a baseline nobody wrote down
print("always no  ", round(1 - year3["dry_3h"].mean(), 3))
print("same as now", round((year3["dry_now"] == year3["dry_3h"]).mean(), 3))

**"Whatever it is now, it will be in three hours" beats our 500-tree model.** On time series, the majority class is the wrong baseline. Ask: what would a lazy forecaster say?

In [ ]:
# ▶ DEMO -- why the random split cheats: test minutes sit between training minutes
hour = df[(df["hpwren_timestamp"] >= "2013-10-05 06:00") & (df["hpwren_timestamp"] < "2013-10-05 07:00")]
is_test = np.random.default_rng(42).random(len(hour)) < 0.2
plt.figure(figsize=(9, 3.5))
plt.plot(hour["hpwren_timestamp"], hour["relative_humidity"], color="lightgray", zorder=1)
plt.scatter(hour["hpwren_timestamp"][~is_test], hour["relative_humidity"][~is_test], label="train", zorder=2)
plt.scatter(hour["hpwren_timestamp"][is_test], hour["relative_humidity"][is_test], color="tab:red", label="test", zorder=3)
plt.ylabel("relative humidity"); plt.legend(); plt.show()

print("typical change in one minute:", round(df["relative_humidity"].diff().abs().median(), 2), "points")
print("next minute has the same dry_3h label:", f"{(df['dry_3h'] == df['dry_3h'].shift(1)).mean():.0%}")

In [ ]:
# ▶ DEMO -- the more capable the model, the more it cheats (about half a minute)
smaller = dict(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
a = XGBClassifier(**smaller).fit(X_tr, y_tr).score(X_te, y_te)
b = XGBClassifier(**smaller).fit(past[SENSORS], past["dry_3h"]).score(year3[SENSORS], year3["dry_3h"])
print(f"depth 6, 300 trees    random split {a:.3f}   final year {b:.3f}")
print(f"depth 12, 500 trees   random split {random_score:.3f}   final year {deep.score(year3[SENSORS], year3['dry_3h']):.3f}")

In [ ]:
# ▶ DEMO -- from here on: three years, three jobs
year1, year2, year3 = three_years(df)
for name, part in [("year 1, train", year1), ("year 2, stop", year2), ("year 3, exam", year3)]:
    print(f"{name:15} {len(part):>8,} rows   dry in 3h {part['dry_3h'].mean():.1%}")

Year 1 trains, year 2 chooses the settings, year 3 is the final test. Whole years, so every season is in every split. And year 3 is drier than the other two: **the future does not promise to look like the past.**

## 8.6 Knobs And Stopping

| Knob | Means | Typical |
| --- | --- | --- |
| `n_estimators` | how many trees | 100 – 3000 |
| `learning_rate` | how much each tree is trusted | 0.01 – 0.3 |
| `max_depth` | how complex each tree is | 3 – 8 |

In [ ]:
# ▶ SETUP 8.6 — from cold
df = weather()
year1, year2, year3 = three_years(df)
past = pd.concat([year1, year2])
print(len(past), "rows in the past   ", len(year3), "in the exam")

In [ ]:
# ▶ DEMO -- max_depth: train on year 1, check on year 2. Year 3 stays untouched (about a minute)
for depth in [2, 3, 6, 9, 12]:
    m = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=depth, random_state=42)
    m.fit(year1[SENSORS], year1["dry_3h"])
    print(f"depth {depth:>2}   year 1 (training) {m.score(year1[SENSORS], year1['dry_3h']):.3f}   year 2 {m.score(year2[SENSORS], year2['dry_3h']):.3f}")

In [ ]:
# ▶ DEMO -- learning_rate against trees: train on year 1, watch year 2 (about two minutes)
for rate, color in [(0.3, "tab:red"), (0.1, "tab:orange"), (0.03, "tab:blue")]:
    m = XGBClassifier(n_estimators=500, learning_rate=rate, max_depth=6, eval_metric="logloss", random_state=42)
    m.fit(year1[SENSORS], year1["dry_3h"], verbose=False,
          eval_set=[(year1[SENSORS], year1["dry_3h"]), (year2[SENSORS], year2["dry_3h"])])
    train_curve = m.evals_result()["validation_0"]["logloss"]
    valid_curve = m.evals_result()["validation_1"]["logloss"]
    best = int(np.argmin(valid_curve))
    print(f"learning_rate {rate:<5} best round {best:>4}   log loss on year 2 {valid_curve[best]:.4f}")
    plt.plot(train_curve, color=color, ls=":")
    plt.plot(valid_curve, color=color, label=f"learning_rate {rate}")
plt.xlabel("trees built"); plt.ylabel("log loss (lower is better)")
plt.title("solid = year 2, dotted = year 1"); plt.legend(); plt.show()

**Low and slow lands slightly better, and takes longer.** Every solid curve turns upward: past its best round, each new tree is fitting noise in year 1. So the real question is not "how many trees?" It is "when do I stop?"

In [ ]:
# ▶ DEMO -- early stopping: ask for 3,000 trees and let year 2 decide
sensors_model = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                              early_stopping_rounds=20, random_state=42)
sensors_model.fit(year1[SENSORS], year1["dry_3h"],
                  eval_set=[(year2[SENSORS], year2["dry_3h"])], verbose=False)
print("stopped at tree", sensors_model.best_iteration)

In [ ]:
# ▶ DEMO -- why depth 3? With early stopping on, compare depths on year 2 only
for depth in [2, 3, 6]:
    m = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=depth, early_stopping_rounds=20,
                      eval_metric="logloss", random_state=42)
    m.fit(year1[SENSORS], year1["dry_3h"], eval_set=[(year2[SENSORS], year2["dry_3h"])], verbose=False)
    print(f"depth {depth}   best tree {m.best_iteration:>4}   log loss {m.best_score:.4f}   accuracy {m.score(year2[SENSORS], year2['dry_3h']):.3f}")

Depths 2 and 3 tie, so we use 3; either would do. **Year 2 made that choice, not year 3.**

**Year 2 is the third split.** Choose when to stop by watching the exam and the exam has helped build the model. And check any code an agent writes for this: most tutorials pass `early_stopping_rounds` to `fit()`, which raises a `TypeError` in current XGBoost. It belongs in the constructor.

## 8.7 Earning It

Tuning has tied the baseline. What can the model not see yet?

In [ ]:
# ▶ SETUP 8.7 — from cold
df = weather()
year1, year2, year3 = three_years(df)
sensors_model = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                              early_stopping_rounds=20, random_state=42)
sensors_model.fit(year1[SENSORS], year1["dry_3h"],
                  eval_set=[(year2[SENSORS], year2["dry_3h"])], verbose=False)

def show(name, y, pred):
    acc, missed, false_alarms = errors(y, pred)
    print(f"{name:28} accuracy {acc:.3f}   missed {missed:>6,}   false alarms {false_alarms:>6,}")

show("same as now", year3["dry_3h"], year3["dry_now"])
show("xgboost, sensors", year3["dry_3h"], sensors_model.predict(year3[SENSORS]))

In [ ]:
# ▶ DEMO -- features from the past: is it drying out? what time is it?
df["hour"]  = df["hpwren_timestamp"].dt.hour
df["month"] = df["hpwren_timestamp"].dt.month
df["rh_change_1h"] = df["relative_humidity"] - df["relative_humidity"].shift(60)
df["rh_change_3h"] = df["relative_humidity"] - df["relative_humidity"].shift(180)
df["temp_change_1h"] = df["air_temp"] - df["air_temp"].shift(60)
df["pressure_change_3h"] = df["air_pressure"] - df["air_pressure"].shift(180)
features = SENSORS + ["hour", "month", "rh_change_1h", "rh_change_3h",
                      "temp_change_1h", "pressure_change_3h"]
print(df[features].isna().sum().max(), "blanks at most in one column -- XGBoost accepts blanks")

In [ ]:
# ▶ DEMO -- the same settings, more to look at
year1, year2, year3 = three_years(df)
model = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                      early_stopping_rounds=20, random_state=42)
model.fit(year1[features], year1["dry_3h"],
          eval_set=[(year2[features], year2["dry_3h"])], verbose=False)
print("stopped at tree", model.best_iteration)
show("same as now", year3["dry_3h"], year3["dry_now"])
show("xgboost, sensors", year3["dry_3h"], sensors_model.predict(year3[SENSORS]))
show("xgboost, plus features", year3["dry_3h"], model.predict(year3[features]))

Better than sensors alone on all three numbers. Against "same as now" it is a trade: 4,646 more missed, 8,567 fewer false alarms. For a fair comparison, make them miss the same number. Normally the model says "dry" at probability 0.5; we lower that until it misses 24,601. This uses year 3's answers, so it compares the two models; you could not set it in advance.

In [ ]:
# ▶ DEMO -- miss exactly as many as the baseline, then count the false alarms
prob = model.predict_proba(year3[features])[:, 1]
prob_on_dry_minutes = prob[year3["dry_3h"] == 1]
threshold = np.quantile(prob_on_dry_minutes, 24601 / len(prob_on_dry_minutes))   # misses what the baseline misses
alarm = prob >= threshold
print("threshold", round(threshold, 3))
show("xgboost, plus features", year3["dry_3h"], alarm)

prob_s = sensors_model.predict_proba(year3[SENSORS])[:, 1]
alarm_s = prob_s >= np.quantile(prob_s[year3["dry_3h"] == 1], 24601 / len(prob_on_dry_minutes))
show("xgboost, sensors", year3["dry_3h"], alarm_s)
show("same as now", year3["dry_3h"], year3["dry_now"])

**3,533 fewer false-alarm minutes than "same as now", or 14%.** Now a threshold you could actually use: choose it on year 2, then apply it to year 3 untouched.

In [ ]:
# ▶ DEMO -- set the threshold on year 2, apply it to year 3
base_y2 = errors(year2["dry_3h"], year2["dry_now"])[1]       # how many same as now misses on year 2
prob_y2 = model.predict_proba(year2[features])[:, 1]
on_dry = prob_y2[year2["dry_3h"] == 1]
threshold_y2 = np.quantile(on_dry, base_y2 / len(on_dry))
print("threshold from year 2:", round(threshold_y2, 3))
show("same as now", year3["dry_3h"], year3["dry_now"])
show("xgboost, plus features", year3["dry_3h"], prob >= threshold_y2)

**2,354 more misses, 6,233 fewer false alarms.** Year 3 is drier than year 2, so the threshold doesn't carry over exactly. A real improvement, and smaller than 0.958 promised. Whether the crew should use it depends on what a miss costs, and that is not in the data.

## 8.8 One More Trap: Rain

Is it raining **this minute**, judging by the other sensors? Not a forecast. Only 0.6% of minutes have rain.

In [ ]:
# ▶ SETUP 8.8 — from cold
df = weather()
df["rain"] = (df["rain_accumulation"] > 0).astype(int)
year1, year2, year3 = three_years(df)
past = pd.concat([year1, year2])
no_rain_cols = [s for s in SENSORS if not s.startswith("rain")]
print("rain minutes in the file:", df["rain"].sum(), "  in year 3:", year3["rain"].sum())

In [ ]:
# ▶ DEMO -- accuracy against what was actually caught (about half a minute)
def rain_row(name, pred):
    acc, missed, false_alarms = errors(year3["rain"], pred)
    print(f"{name:32} accuracy {acc:.3f}   caught {year3['rain'].sum() - missed:>4}   false alarms {false_alarms:>6,}")

rain_row('always "no"', np.zeros(len(year3), int))
for weight in [1, 150]:
    m = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, scale_pos_weight=weight, random_state=42)
    m.fit(past[no_rain_cols], past["rain"])
    rain_row(f"xgboost, scale_pos_weight={weight}", m.predict(year3[no_rain_cols]))

`scale_pos_weight=150` counts each rainy minute 150 times in training. It catches nine times the rain for thirty times the false alarms, and accuracy falls from 0.995 to 0.959. **At this ratio accuracy is useless: the model that does nothing scores highest.** Report what was caught and what it cost.

## 8.9 Your Turn — 15 minutes

Six hours ahead, on the same exam. Agent on; you will be asked to explain any cell.

In [ ]:
# ✍️ YOU TRY 8.9
# 1. rows_ahead  -- how many rows ahead is six hours?
# 2. d           -- weather() plus a 0/1 column dry_6h. Drop the rows with no answer yet.
#                   Also add one column of your own that looks only BACKWARD; name it in my_feature.
# 3. year1, year2, year3 = three_years(d)
# 4. always_no, same_as_now  -- accuracy of each baseline on year 3, against dry_6h
# 5. stopped     -- XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
#                   early_stopping_rounds=20, random_state=42), on SENSORS, stopped by year 2
#    sensors_acc -- its accuracy on year 3
# 6. mine        -- the same settings on SENSORS + [my_feature]
#    my_acc, my_missed, my_false = errors(year3["dry_6h"], mine.predict(...))

rows_ahead = 0                                # TODO
d = weather()                                 # TODO: dry_6h, your feature, drop the unanswered rows
my_feature = ""                               # TODO
always_no = same_as_now = sensors_acc = 0.0   # TODO
stopped = None                                # TODO
my_acc, my_missed, my_false = 0.0, 0, 0       # TODO

print(rows_ahead, my_feature)
print(round(always_no, 3), round(same_as_now, 3), round(sensors_acc, 3))
print(round(my_acc, 3), my_missed, my_false)

In [ ]:
# ✅ CHECKPOINT 8.9
assert rows_ahead == 360, rows_ahead
assert abs(always_no - 0.697) < 0.002, always_no
assert abs(same_as_now - 0.872) < 0.002, same_as_now
assert stopped is not None and 50 < stopped.best_iteration < 300, "fit `stopped` with early stopping on year 2"
assert abs(sensors_acc - 0.876) < 0.005, sensors_acc
assert my_feature in d.columns and my_feature not in SENSORS, my_feature
assert 0.85 < my_acc < 0.93, ("above 0.93 usually means your feature looks forward", my_acc)
base_acc, base_missed, base_false = errors(year3["dry_6h"], year3["dry_now"])
print("Correct.\n")
print(f"same as now   accuracy {base_acc:.3f}   missed {base_missed:,}   false alarms {base_false:,}")
print(f"your model    accuracy {my_acc:.3f}   missed {my_missed:,}   false alarms {my_false:,}")

**Write your two sentences here** (double-click to edit):

> Would you give this model to the controlled-burn crew? One sentence with the evidence: your model against the better baseline, on year 3, in misses and false alarms. One on what would make you change your answer.

*(your answer)*

## Wrap

You can now:

- Write gradient boosting's loop, and say how it differs from a forest
- Fit XGBoost, tune three knobs, and stop it with a held-out year
- Compare models with cross-validation rather than one lucky split
- Split time series by time, and say why a random split cheats
- Pick a baseline that a lazy forecaster would actually use
- Say why 99.6% accuracy can be a failing grade

**Next:** more ways to get a model on this same file. Keep your year 3 numbers.